# 14. Add label-preserving formatting augmentation

Owner: Jalil-g (reassigned from Gtpatel02 on November 10, 2026). Issue #14. Depends on completed #12 and #13.

## Goal

#12 showed formatting changes hurt our models, and #13 leans on blank lines. Here we add whitespace-only copies of training rows to the #13 HistGB style model and check if it holds up better on reformatted text without losing original-text AUROC.

## Rule (fixed before running)

**Presets.** `original_only`; `plus_collapse` (original + collapsed whitespace); `plus_newline_wrap` (original + single newline + wrap64 + wrap80). Originals get weight 1, each copied view weight 0.25. Every fit row gets every copy of its preset, even when the view leaves the text unchanged, so each source row has the same total weight and classes stay balanced.

**Views.** Only whitespace changes; the word sequence `text.split()` stays identical, so labels are kept.
- collapse: `" ".join(text.split())`.
- single newline: every blank-line separator (one or more whitespace-only lines) becomes one `\n`.
- wrap64 / wrap80: split into chunks on blank lines, `textwrap.fill` each chunk at the width with `break_long_words=False, break_on_hyphens=False`, join chunks with one blank line.

**Selection.** Internal split: 6400 fit / 1600 eval inside original8000, seed 42, exact 800/800 (same IDs as #12 split42). Score each preset on the 1600 eval rows under original, collapse, single newline, wrap64, wrap80. Baseline = `original_only` on original text. Eligible presets have original AUROC >= baseline - 0.005. Pick the highest mean of original/collapse/wrap80 AUROC. Exact tie goes to fewer copies. The choice is saved before frozen validation is touched.

**Confirmation.** Refit all three presets on the 8000 development rows (same views, same weights), score frozen 2000 under all views. Only the saved choice is the handoff. Validation was reused before (#4, #8, #9, #13), so it is optimistic.

Caveat known up front: `plus_collapse` trains on the collapse view and `plus_newline_wrap` trains on wrap80, so each preset has seen part of the selection views. A copied view is not a new independent labeled example; it is the same row in different whitespace.

## Setup

Reuse the #13 feature code with `RICH_STYLE_HELPERS_ONLY=1`, so nothing from #13 fits or writes. HistGB settings come from the completed #13 record. One OpenMP thread, as in #13.

In [ ]:
import json, os, re, textwrap, time, traceback
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
os.environ["RICH_STYLE_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./13_rich_style.ipynb")

NAME = "14_style_augmentation"
COMMAND = "uv run --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=1800 notebooks/14_style_augmentation.ipynb"
INNER_EVAL_PER_CLASS = 800
COPY_WEIGHT = 0.25
ORIGINAL_SLACK = 0.005
PRESETS = {"original_only": [], "plus_collapse": ["collapse"],
           "plus_newline_wrap": ["single_newline", "wrap64", "wrap80"]}  # order = tie order, fewer copies first
SELECTION_VIEWS = ["original", "collapse", "wrap80"]

BLANK_SEPARATOR = re.compile(r"\n\s*\n")  # greedy: one or more whitespace-only lines between two lines

def collapse(text):
    return " ".join(str(text).split())

def single_newline(text):
    return BLANK_SEPARATOR.sub("\n", str(text))

def wrap_view(text, width):
    chunks = [c for c in PARAGRAPH_SPLIT.split(str(text)) if c.strip()]
    return "\n\n".join(textwrap.fill(c, width=width, break_long_words=False, break_on_hyphens=False) for c in chunks)

VIEWS = {"original": str, "collapse": collapse, "single_newline": single_newline,
         "wrap64": lambda t: wrap_view(t, 64), "wrap80": lambda t: wrap_view(t, 80)}

## Checks

Notebook 02 checks data bytes, duplicate audit, and the frozen split. #13 must have a completed record whose notebook source still matches. The inner split must match #12 split42 digests. View functions are tested on synthetic strings, then we check on all 10000 labeled texts that no view changes the word sequence.

In [ ]:
START = time.perf_counter()
parts = load_partitions()
train, validation = parts["train"], parts["validation"]
records = [(p, json.loads(p.read_text())) for p in sorted((ROOT/"results/13_rich_style").glob("2*Z.json"))]
records = [(p, r) for p, r in records if r.get("status") == "completed"]
require(bool(records), "Completed #13 is required")
dep13_path, dep13 = records[-1]
nb13_hash = value_hash(["".join(c["source"]) for c in json.loads((ROOT/"notebooks/13_rich_style.ipynb").read_text())["cells"]])
require(nb13_hash == dep13["config"]["notebook_source_sha256"], "Notebook 13 source changed since its recorded run")
require(dep13["config"]["schema_sha256"] == SCHEMA_SHA256 and dep13["metrics"]["selected"] == "histgb", "Unexpected #13 schema or choice")
HISTGB = dep13["config"]["candidates"]["histgb"]
dep12_path = sorted((ROOT/"results/12_seed_stress").glob("2*Z.json"))[-1]
dep12 = json.loads(dep12_path.read_text())
require(dep12["status"] == "completed", "Completed #12 is required")

_, components, _ = checked_audit(ROOT / parts["split"]["audit_path"])
train_ids = set(train.id)
inner_components = [[row for row in comp if row["file"] == "train.csv" and row["id"] in train_ids] for comp in components]
inner_components = [comp for comp in inner_components if len(comp) > 1]
fit_ids, eval_ids = split_ids(train[["id", "label"]], inner_components, validation_per_class=INNER_EVAL_PER_CLASS, seed=SEED)
split_digest = {"eval_ids_sha256": value_hash(eval_ids), "fit_ids_sha256": value_hash(fit_ids)}
require(split_digest == dep12["config"]["inner_split_digests"]["42"], "Inner split differs from #12 split42")
indexed = train.set_index("id", drop=False)
inner_fit, inner_eval = indexed.loc[fit_ids].reset_index(drop=True), indexed.loc[eval_ids].reset_index(drop=True)
require(len(inner_fit) == 6400 and inner_eval.label.value_counts().to_dict() == {0: 800, 1: 800}, "Bad inner split")

# Synthetic view checks.
sample = "Title line\n\n\nFirst  para has a well-known hyphen-word and " + "word " * 20 + "\n  \nSecond\tpara\nnext line " + "x" * 90
require("\n" not in collapse(sample) and "  " not in collapse(sample), "Collapse failed")
require(not re.search(r"\n\s*\n", single_newline(sample)) and single_newline("a\n\n\nb\n  \nc\nd") == "a\nb\nc\nd", "Single newline failed")
for width in [64, 80]:
    wrapped = wrap_view(sample, width)
    require(wrapped.count("\n\n") == 2 and all(len(line) <= width or " " not in line for line in wrapped.split("\n")), f"wrap{width} failed")
    require("hyphen-word" in wrapped and "x" * 90 in wrapped, f"wrap{width} broke a word")
require(all(VIEWS[v]("") == "" for v in VIEWS), "Empty text failed")

# Real-text check: every view keeps the exact word sequence, so labels stay valid.
labeled = pd.concat([train, validation])
changed = {}
for view, change in VIEWS.items():
    texts = labeled.text.map(change)
    require((texts.str.split() == labeled.text.str.split()).all(), f"{view} changed words")
    changed[view] = int((texts != labeled.text).sum())
CONFIG = {"histgb": HISTGB, "schema_sha256": SCHEMA_SHA256, "presets": PRESETS, "copy_weight": COPY_WEIGHT,
    "views": {"original": "unchanged", "collapse": "' '.join(text.split())",
              "single_newline": "re.sub(r'\\n\\s*\\n', '\\n')",
              "wrap64": "split on blank lines; textwrap.fill width 64, break_long_words=False, break_on_hyphens=False; join with blank line",
              "wrap80": "same as wrap64, width 80"},
    "copies": "every fit row gets every copy of its preset, changed or not",
    "selection": "eligible if original >= original_only original - 0.005; max mean AUROC over original/collapse/wrap80 on inner 1600; tie to fewer copies; saved before validation",
    "original_slack": ORIGINAL_SLACK, "selection_views": SELECTION_VIEWS,
    "inner_split": {"fit": 6400, "eval": 1600, "seed": SEED, **split_digest, "matches_12_split42": True},
    "views_changed_rows_10000": changed, "threads": "HistGB limited to 1 OpenMP thread",
    "dependencies": {"13": {"record": str(dep13_path.relative_to(ROOT)), "record_sha256": file_hash(dep13_path), "source_sha256": nb13_hash},
                     "12": {"record": str(dep12_path.relative_to(ROOT)), "record_sha256": file_hash(dep12_path)}},
    "notebook_source_sha256": value_hash(["".join(c["source"]) for c in json.loads((ROOT/"notebooks/14_style_augmentation.ipynb").read_text())["cells"]])}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
private = ROOT/"artifacts"/NAME/RUN_ID
private.mkdir(parents=True, exist_ok=False)
print("Rows changed by each view, out of 10000 labeled:", changed)
print("Inner split matches #12 split42. Checks passed.")

## Method

Features are deterministic and learn nothing from labels, so each view of a row is featurized once. Only HistGB learns, and only on fit rows plus their copies. Eval and validation views are used for scoring only.

In [ ]:
def view_features(frame):
    return {view: rich_style_features(frame.text.map(change)) for view, change in VIEWS.items()}

def fit_preset(preset, feats, labels):
    views = ["original"] + PRESETS[preset]
    X = np.vstack([feats[v] for v in views])
    y = np.concatenate([labels] * len(views))
    weights = np.concatenate([np.ones(len(labels))] + [np.full(len(labels), COPY_WEIGHT)] * len(PRESETS[preset]))
    with threadpool_limits(limits=1, user_api="openmp"):
        return HistGradientBoostingClassifier(**HISTGB).fit(X, y, sample_weight=weights)

def score_views(model, feats, labels):
    return {view: float(roc_auc_score(labels, class1_probabilities(model, X))) for view, X in feats.items()}

def select(scores):
    baseline = scores["original_only"]["original"]
    rows = [{"preset": p, "original": s["original"], "eligible": s["original"] >= baseline - ORIGINAL_SLACK,
             "selection_mean": float(np.mean([s[v] for v in SELECTION_VIEWS]))} for p, s in scores.items()]
    eligible = [r for r in rows if r["eligible"]]
    best = max(r["selection_mean"] for r in eligible)
    return next(r["preset"] for r in eligible if r["selection_mean"] == best), baseline, rows  # PRESETS order breaks ties

## Run

Save the plan, fit the three presets on 6400 and score 1600, save the choice, then refit all three on 8000 and score frozen 2000. A failure writes a safe failed record and keeps the traceback private.

In [ ]:
plan_path = ROOT/"results"/NAME/f"{RUN_ID}-plan.json"
save_new_json(plan_path, {"config": CONFIG, "config_sha256": value_hash(CONFIG), "saved_before_fit": True, "split_sha256": parts["split"]["split_sha256"]})
artifacts, timings, models = [plan_path], [], {}
try:
    t0 = time.perf_counter()
    inner_fit_feats, inner_eval_feats = view_features(inner_fit), view_features(inner_eval)
    timings.append({"stage": "inner_features", "seconds": time.perf_counter() - t0})
    inner_scores = {}
    for preset in PRESETS:
        t0 = time.perf_counter()
        models[("internal6400", preset)] = fit_preset(preset, inner_fit_feats, inner_fit.label.to_numpy())
        timings.append({"stage": f"internal6400_fit_{preset}", "seconds": time.perf_counter() - t0})
        inner_scores[preset] = score_views(models[("internal6400", preset)], inner_eval_feats, inner_eval.label.to_numpy())
    selected, baseline, selection_rows = select(inner_scores)
    selection_path = ROOT/"results"/NAME/f"{RUN_ID}-selection.json"
    save_new_json(selection_path, {"selected": selected, "baseline_original_auroc": baseline, "rows": selection_rows,
                                   "inner_auroc": inner_scores, "saved_before_validation": True})
    artifacts.append(selection_path)

    t0 = time.perf_counter()
    dev_feats, val_feats = view_features(train), view_features(validation)
    timings.append({"stage": "development_features", "seconds": time.perf_counter() - t0})
    val_scores = {}
    for preset in PRESETS:
        t0 = time.perf_counter()
        models[("development8000", preset)] = fit_preset(preset, dev_feats, train.label.to_numpy())
        timings.append({"stage": f"development8000_fit_{preset}", "seconds": time.perf_counter() - t0})
        val_scores[preset] = score_views(models[("development8000", preset)], val_feats, validation.label.to_numpy())
    run_seconds = time.perf_counter() - START
except Exception as exc:
    (private/"failure_traceback.txt").write_text(traceback.format_exc())
    record_run(NAME, COMMAND, CONFIG, SEED, 0, 0, {"validation_auroc": None}, time.perf_counter() - START,
               artifacts=artifacts, status="failed", error_type=type(exc).__name__)
    raise RuntimeError("Augmentation run failed. See safe record and ignored failure_traceback.txt.") from None

## Results

Inner rows are 1600 eval rows from original8000. Validation is frozen 2000, reused before, so optimistic. Not Kaggle scores.

In [ ]:
inner_table = pd.DataFrame(inner_scores).T[list(VIEWS)]
val_table = pd.DataFrame(val_scores).T[list(VIEWS)]
print("Inner 6400/1600 AUROC by preset (rows) and scored view (columns):")
display(inner_table.round(6))
display(pd.DataFrame(selection_rows).round(6))
print(f"Selected before validation: {selected}. Baseline original AUROC {baseline:.6f}, floor {baseline - ORIGINAL_SLACK:.6f}.")
print("Frozen 2000 validation AUROC, all presets refit on 8000:")
display(val_table.round(6))
loss = pd.concat({"inner": inner_table.rsub(inner_table["original"], axis=0), "validation": val_table.rsub(val_table["original"], axis=0)})
print("Loss = original AUROC minus view AUROC, same model:")
display(loss.drop(columns="original").round(6))
display(pd.DataFrame(timings).round(2))
print(f"Whole run: {run_seconds:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (title, table) in zip(axes, [("Inner 1600 (fit 6400)", inner_table), ("Frozen 2000 (fit 8000)", val_table)]):
    table.T.plot(ax=ax, marker="o")
    ax.set(title=title, xlabel="scored view", ylabel="AUROC")
fig.tight_layout()
plot_path = ROOT/"results"/NAME/f"{RUN_ID}-views.png"
fig.savefig(plot_path, dpi=130)
artifacts.append(plot_path)
plt.show()

## Takeaway

- Tried: three whitespace-copy presets on the #13 HistGB style model, copies at weight 0.25, rule fixed before the run. Inner 6400/1600 picked the preset, frozen 2000 confirmed.
- Found: `plus_collapse` won (inner mean 0.971030 vs 0.949265 newline+wrap, 0.919932 original only) and cost little original AUROC (inner -0.000506, validation -0.000343). It cut collapse loss from 0.188 to 0.053 inner. Each preset mostly fixes only the views it trained on: newline+wrap keeps wrap80 loss under 0.008 but collapse loss stays 0.13.
- Next: hand the 8000-row `plus_collapse` model to #17. A preset with all four copies was not tested; trying it now would be a post-hoc pick on the same rows.

`original_only` on 8000 gives validation 0.998105, same as #13, so the refit path matches.

## Save

All six models stay under ignored `artifacts/14_style_augmentation/`. A private pointer `artifacts/14_style_augmentation/latest.json` names the selected 8000-row model for #17. The safe record keeps every arm on every view, the selection, and timings. No row predictions are saved and nothing is uploaded.

In [ ]:
reload_difference = {}
for (stage, preset), model in models.items():
    path = private/f"{stage}_{preset}.joblib"
    joblib.dump(model, path)
    feats = inner_eval_feats if stage == "internal6400" else val_feats
    reload_difference[f"{stage}_{preset}"] = float(np.max(np.abs(class1_probabilities(joblib.load(path), feats["original"])
                                                                  - class1_probabilities(model, feats["original"]))))
    require(reload_difference[f"{stage}_{preset}"] == 0, "Reloaded model changed predictions")
    artifacts.append(path)
metrics = {"validation_auroc": val_scores[selected]["original"], "selected": selected,
           "baseline_original_auroc": baseline, "selection_rows": selection_rows,
           "internal6400_eval1600_auroc": inner_scores, "development8000_validation2000_auroc": val_scores,
           "timings": timings, "max_reload_difference": reload_difference}
CONFIG["extra_versions"] = {name: importlib.metadata.version(name) for name in ["joblib", "matplotlib", "threadpoolctl"]}
record_path = record_run(NAME, COMMAND, CONFIG, SEED, len(train), len(validation), metrics,
                         time.perf_counter() - START, artifacts=artifacts)
pointer = {"result": str(record_path.relative_to(ROOT)), "result_sha256": file_hash(record_path),
           "split_sha256": parts["split"]["split_sha256"], "selected": selected,
           "model": str((private/f"development8000_{selected}.joblib").relative_to(ROOT)),
           "schema_sha256": SCHEMA_SHA256,
           "feature_interface": "class1_probabilities(joblib.load(model), rich_style_features(texts)) with RICH_STYLE_HELPERS_ONLY=1 notebook 13",
           "artifacts": {str(Path(p).relative_to(ROOT)): file_hash(p) for p in artifacts}}
(ROOT/"artifacts"/NAME/"latest.json").write_text(json.dumps(pointer, indent=2, sort_keys=True) + "\n")
print(f"Saved {record_path.relative_to(ROOT)}")